In [0]:
--crosswalk 
DESCRIBE TABLE pondolens.dev_silver.shared_region_crosswalk;

SELECT *
FROM pondolens.dev_silver.shared_region_crosswalk
ORDER BY source_region;

SELECT
  source_region,
  region_code,
  COUNT(*) AS row_count
FROM pondolens.dev_silver.shared_region_crosswalk
GROUP BY source_region, region_code
ORDER BY source_region, region_code;


CREATE OR REPLACE TABLE pondolens.dev_silver.shared_region_crosswalk AS
SELECT *
FROM VALUES
  ('BARMM', 'BARMM', 'BARMM'),
  ('Cordillera Administrative Region', 'CAR', 'Cordillera Administrative Region'),
  ('MIMAROPA Region', 'IVB', 'MIMAROPA'),
  ('NCR - National Capital Region', 'NCR', 'National Capital Region'),
  ('NCR National Capital Region', 'NCR', 'National Capital Region'),
  ('Region I - Ilocos Region', 'I', 'Ilocos Region'),
  ('Region I Ilocos Region', 'I', 'Ilocos Region'),
  ('Region II - Cagayan Valley', 'II', 'Cagayan Valley'),
  ('Region II Cagayan Valley', 'II', 'Cagayan Valley'),
  ('Region III - Central Luzon', 'III', 'Central Luzon'),
  ('Region III Central Luzon', 'III', 'Central Luzon'),
  ('Region IV A CALABARZON', 'IVA', 'CALABARZON'),
  ('Region IV-A CALABARZON', 'IVA', 'CALABARZON'),
  ('Region V - Bicol Region', 'V', 'Bicol Region'),
  ('Region V Bicol Region', 'V', 'Bicol Region'),
  ('Region VI - Western Visayas', 'VI', 'Western Visayas'),
  ('Region VI Western Visayas', 'VI', 'Western Visayas'),
  ('Region VII - Central Visayas', 'VII', 'Central Visayas'),
  ('Region VII Central Visayas', 'VII', 'Central Visayas'),
  ('Region VIII - Eastern Visayas', 'VIII', 'Eastern Visayas'),
  ('Region VIII Eastern Visayas', 'VIII', 'Eastern Visayas'),
  ('Region IX - Zamboanga Peninsula', 'IX', 'Zamboanga Peninsula'),
  ('Region IX Zamboanga Peninsula', 'IX', 'Zamboanga Peninsula'),
  ('Region X - Northern Mindanao', 'X', 'Northern Mindanao'),
  ('Region X Northern Mindanao', 'X', 'Northern Mindanao'),
  ('Region XI - Davao Region', 'XI', 'Davao Region'),
  ('Region XI Davao Region', 'XI', 'Davao Region'),
  ('Region XII - SOCCSKSARGEN', 'XII', 'SOCCSKSARGEN'),
  ('Region XII SOCCSKSARGEN', 'XII', 'SOCCSKSARGEN'),
  ('Region XIII - Caraga', 'XIII', 'Caraga'),
  ('Region XIII Caraga', 'XIII', 'Caraga')
AS crosswalk(source_region_name, region_code, region_name);


SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT source_region_name) AS source_names,
  COUNT(DISTINCT region_code) AS region_codes,
  COUNT(DISTINCT region_name) AS standard_region_names
FROM pondolens.dev_silver.shared_region_crosswalk;


SELECT *
FROM pondolens.dev_silver.shared_region_crosswalk
ORDER BY region_code, source_region_name;


DESCRIBE TABLE pondolens.dev_bronze.hospital;


SELECT
  h.region_name AS source_region_name,
  x.region_code,
  x.region_name AS standard_region_name,
  COUNT(*) AS row_count,
  SUM(CASE WHEN TRY_CAST(TRIM(h.year) AS INT) IS NULL THEN 1 ELSE 0 END)
    AS invalid_year_rows,
  SUM(CASE WHEN TRY_CAST(TRIM(h.regulated_hospital_count) AS INT) IS NULL THEN 1 ELSE 0 END)
    AS invalid_count_rows
FROM pondolens.dev_bronze.hospital h
LEFT JOIN pondolens.dev_silver.shared_region_crosswalk x
  ON TRIM(h.region_name) = x.source_region_name
GROUP BY
  h.region_name,
  x.region_code,
  x.region_name
ORDER BY
  source_region_name;

SELECT
  year,
  COUNT(*) AS row_count,
  COUNT(DISTINCT region_name) AS region_count,
  COLLECT_SET(hospital_type) AS hospital_types,
  COLLECT_SET(unit) AS units
FROM pondolens.dev_bronze.hospital
GROUP BY year
ORDER BY year;


SELECT
  year,
  region_name,
  hospital_type,
  COUNT(*) AS duplicate_count
FROM pondolens.dev_bronze.hospital
GROUP BY year, region_name, hospital_type
HAVING COUNT(*) > 1
ORDER BY year, region_name, hospital_type;

SELECT
  source_region_name,
  COUNT(*) AS row_count,
  COUNT(DISTINCT region_code) AS region_code_count,
  COUNT(DISTINCT region_name) AS region_name_count
FROM pondolens.dev_silver.shared_region_crosswalk
GROUP BY source_region_name
HAVING COUNT(*) <> 1
    OR COUNT(DISTINCT region_code) <> 1
    OR COUNT(DISTINCT region_name) <> 1;


  ------------------------------------------------------------------
--Query for Hospital



